In [14]:
import pandas as pd
import numpy as np
%pip install openpyxl

Note: you may need to restart the kernel to use updated packages.


In [22]:
df = pd.read_excel('problem1.xlsx')

In [23]:
df

,multipacity,age,male,female
0,Copenhagen,0 years,5039,4882
1,Copenhagen,1 year,4412,4178
2,Copenhagen,2 years,4104,3940
3,Copenhagen,3 years,3717,3590
4,Copenhagen,4 years,3391,3216
...,...,...,...,...
3775,Christiansø,121 years,0,0
3776,Christiansø,122 years,0,0
3777,Christiansø,123 years,0,0
3778,Christiansø,124 years,0,0


In [25]:
import pandas as pd

df = pd.read_excel("problem1.xlsx")
df["age"] = df["age"].str.extract(r"(\d+)").astype(int)
df["year"] = 2020 - df["age"]

df_long = df.melt(id_vars=["multipacity", "year"], value_vars=["male", "female"],
                  var_name="sex", value_name="count")

percent = df_long.loc[df_long["count"] == 1, "count"].sum() / df_long["count"].sum() * 100
print(round(percent, 3))

0.008


In [53]:
import pandas as pd
import numpy as np

# 1) Načítaj
df = pd.read_excel("problem1.xlsx")

# 2) Rok narodenia (vek je text ako "1 year"/"2 years")
df["age"] = df["age"].str.extract(r"(\d+)").astype(int)
df["year"] = 2020 - df["age"]

# 3) Dlhý tvar: (obec, rok, pohlavie, počet)
d = df.melt(id_vars=["multipacity", "year"],
            value_vars=["male", "female"],
            var_name="sex", value_name="n")

# 4) f(n) = n * (364/365)^(n-1), pri n=0 → 0
ratio = 364.0/365.0
n = d["n"].astype(float)
fn = np.where(n > 0, n * (ratio ** (n - 1.0)), 0.0)

# 5) percento unikátne identifikovateľných (narodeniny+obec+pohlavie)
percent = fn.sum() / n.sum() * 100
print(f"{percent:.2f}")

30.66


In [54]:
###2


In [55]:
# --- Importy ---
import pandas as pd

# --- 1. Načítanie dát ---
# Súbor si načítame, Pandas automaticky zistí oddeľovač
df = pd.read_csv("problem2.csv")

# --- 2. Rýchly náhľad na dáta ---
df.head()

,age,sex,region,num_children,citizenship,criminal_record
0,35,female,Region of Southern Denmark,0,Danish,no
1,48,female,Region of Southern Denmark,0,Danish,yes
2,65,female,North Jutland Region,1,Danish,no
3,18,female,Region of Southern Denmark,1,Other EU,no
4,51,male,Region Zealand,0,Danish,no


In [56]:
# --- 4. Spočítanie frekvencií každej kombinácie (region, age) ---
freqs = df.groupby(["region", "age"]).size().reset_index(name="count")

# --- 5. Vyfiltrovanie len tých, ktoré sa vyskytujú práve raz ---
sample_uniques = freqs[freqs["count"] == 1]

# --- 6. Výsledok: počet sample uniques ---
num_sample_uniques = len(sample_uniques)

print(f"Počet sample uniques (unikátne kombinácie region + age): {num_sample_uniques}")

Počet sample uniques (unikátne kombinácie region + age): 90


In [57]:
# --- 1. Spočítame frekvencie každej kombinácie QI ---
freqs = df.groupby(["sex", "citizenship", "num_children"]).size().reset_index(name="count")

# --- 2. Spojíme tieto frekvencie späť k pôvodným dátam ---
df_merged = df.merge(freqs, on=["sex", "citizenship", "num_children"], how="left")

# --- 3. Pre každý riadok vypočítame individuálne riziko (1/freq) ---
df_merged["risk"] = 1 / df_merged["count"]

# --- 4. Vypočítame priemerné riziko ---
avg_risk = df_merged["risk"].mean()

# --- 5. Zaokrúhlenie na 3 desatinné miesta ---
avg_risk_rounded = round(avg_risk, 3)

print(f"Average reidentification risk (rounded to 3 decimals): {avg_risk_rounded}")

Average reidentification risk (rounded to 3 decimals): 0.113


In [58]:
import pandas as pd

# --- 1. Recode citizenship ---
df["citizenship_recode"] = df["citizenship"].apply(
    lambda x: "Danish" if str(x).strip().lower() in ["denmark", "danish"] else "Non-Danish"
)

# --- 2. Skupiny podľa QI: (region, citizenship_recode) ---
group_sizes = df.groupby(["region", "citizenship_recode"]).size().reset_index(name="count")

# --- 3. Najmenšia skupina = najväčšie k, ktoré dataset spĺňa ---
k = group_sizes["count"].min()

print(f"Largest k for which dataset satisfies k-anonymity: {k}")

Largest k for which dataset satisfies k-anonymity: 4


In [59]:
import pandas as pd

# 1) Recode citizenship → Danish / Non-Danish
def recode_cit(x):
    x = str(x).strip().lower()
    return "Danish" if x in {"denmark", "danish"} else "Non-Danish"

df["citizenship_recode"] = df["citizenship"].apply(recode_cit)

# 2) Normalizácia citlivého atribútu (pre spoľahlivé počítanie distinct)
#    - ak máš True/False/Yes/No, toto ich prevedie na jednotný text
cr_norm = df["criminal_record"].astype(str).str.strip().str.lower().replace({"nan": pd.NA})
df["_criminal_record_norm"] = cr_norm

# 3) Spočítame v QI-skupinách (region, citizenship_recode) počet distinct hodnôt criminal_record
grp = (
    df.groupby(["region", "citizenship_recode"])
      .agg(
          group_size=("criminal_record", "size"),
          cr_n_unique=("_criminal_record_norm", pd.Series.nunique),
          # pre info: ktorá hodnota je unikátna, ak je skupina homogénna
          cr_first=("_criminal_record_norm", "first")
      )
      .reset_index()
)

# 4) Označíme homogénne QI-skupiny (n_unique == 1)
homogeneous_groups = grp[grp["cr_n_unique"] == 1][["region", "citizenship_recode", "group_size", "cr_first"]]

# 5) Pripojíme späť k riadkom, aby sme spočítali, koľko záznamov je odhaliteľných
df_flag = df.merge(
    homogeneous_groups,
    on=["region", "citizenship_recode"],
    how="left",
    indicator=False
)

# riadok je odhaliteľný, ak patrí do homogénnej skupiny (tj. group_size sa naplnila merge-om)
df_flag["attribute_disclosed"] = df_flag["group_size"].notna()

n_disclosed = int(df_flag["attribute_disclosed"].sum())
N = len(df_flag)
pct_disclosed = n_disclosed / N if N else 0.0

print(f"Records with certain knowledge of criminal_record (attribute disclosure): {n_disclosed} / {N} "
      f"({pct_disclosed:.3%})")

# Voliteľne: ukáž TOP pár homogénnych skupín (kde dochádza k odhaleniu) s odhalenou hodnotou
display_cols = ["region", "citizenship_recode", "group_size", "cr_first"]
print("\nHomogeneous QI-groups (criminal_record fully predictable):")
print(homogeneous_groups.sort_values(["group_size", "region"]).to_string(index=False))

Records with certain knowledge of criminal_record (attribute disclosure): 18 / 150 (12.000%)

Homogeneous QI-groups (criminal_record fully predictable):
                    region citizenship_recode  group_size cr_first
       Capital City Region         Non-Danish           4       no
        Mid Jutland Region         Non-Danish           5       no
Region of Southern Denmark         Non-Danish           9       no
